In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_code    = os.path.join(path_git, 'Python Code', 'BLS')
    
    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'BLS')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BLS Data')
    path_main = os.path.join(path_sp, 'Data')
    
if user in ['jchoy', 'aazawii']:
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'BLS', 'config')

print(user)
print(path_git)

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

# Base URL for API V2
url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

# Set API key
exec(open(os.path.join(path_config, 'api_key.txt')).read())
dict_api[user]

In [ ]:
# Execute script to prepare API request inputs
exec(open(os.path.join(path_code, 'Step 01 - Supplemental Scripts', 'Step 01a - Prepare API Request Inputs.py')).read())

# Import objects
df_indicators = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Indicators')
df_indicators = df_indicators[df_indicators['Indicator Name'] == indicator_name]

export_loc = df_indicators['Export Location'].values[0]
folder     = df_indicators['Folder'         ].values[0]

print('')
print('Export location file path: ' + export_loc)
print('Folder name:               ' + folder    )
print('')

In [ ]:
# Import data by geography
path_in = os.path.join(path_main, export_loc, indicator_name + ' ' + folder)

if indicator_name == 'Jobs_2':
    df_jobs2_mpo = pd.read_excel(os.path.join(path_in, 'Jobs_2 MPO BLS SM.xlsx'     ), sheet_name = 'MPO'     )
    df_jobs2_nat = pd.read_excel(os.path.join(path_in, 'Jobs_2 National BLS CE.xlsx'), sheet_name = 'National')
    display(df_jobs2_mpo.head(3), df_jobs2_nat.head(3))


if indicator_name == 'Jobs_3':
    try: 
        df_msa1 = pd.read_excel(os.path.join(path_in, indicator_name + ' MSA BLS ' + survey + '.xlsx'), sheet_name = 'Government and Private')
        df_msa2 = pd.read_excel(os.path.join(path_in, indicator_name + ' MSA BLS ' + survey + '.xlsx'), sheet_name = 'Goods and Services')
        df_msa1 = df_msa1[df_msa1['Sector'] != 'All']
        df_msa2 = df_msa2[df_msa2['Sector'] != 'All']
        df_msa = pd.concat([df_msa1, df_msa2])
        display(df_msa.head(3))
    except Exception as e: print(e)


if indicator_name == 'Labor_2':
    df_msa  = pd.read_excel(os.path.join(path_in, indicator_name + ' MSA BLS ' + survey + '.xlsx'), sheet_name = 'MSA')
    df_jobs = pd.read_excel(os.path.join(path_main, 'Vibrant and Inclusive Places', 'Economy', 'Jobs', 'Jobs_1 Total', 'Jobs_1 MSA BLS SM.xlsx'), sheet_name = 'MSA')
    display(df_msa.head(3), df_jobs.head(3))


In [ ]:
if indicator_name == 'Labor_2':
    df_jobs = df_jobs[df_jobs['Sector'] == 'All']
    df_jobs = df_jobs[['date_', 'area_code', 'Total Jobs']].rename(columns = {'area_code':'MSA_ID'})
    df_msa = df_msa.merge(df_jobs, on = ['date_', 'MSA_ID'], how = 'left')
    df_msa = df_msa[~df_msa['Total Jobs'].isna()]

    conditions = [
        (  df_msa['MSA'] ==  'Sacramento--Roseville--Arden-Arcade, CA Metropolitan Statistical Area') ,
        (  df_msa['MSA'] ==  'Yuba City, CA Metropolitan Statistical Area'                          ) ,
        ( ~df_msa['MSA'].str.contains('Yuba|Sacramento')                                            )
    ]
    choices = ['Sacramento--Roseville--Arden-Arcade, CA Metropolitan Statistical Area', 'Yuba City, CA Metropolitan Statistical Area', 'Peer MSA']
    df_msa["Group"] = np.select(conditions, choices)
    
    df_msa['Year'] = pd.to_datetime(df_msa['date_'])
    df_msa['Year'] = df_msa['Year'].dt.year

    wm = lambda x: np.average(x, weights = df_msa.loc[x.index, "Total Jobs"]) # weighted average
    df_msa = df_msa.groupby(['Year', 'Group'], as_index = False).agg(unemployment_rate = ('Unemployment Rate', wm))
    df_msa['unemployment_rate'] = round(df_msa['unemployment_rate'], 1)
    df_msa = df_msa.sort_values(['Year', 'Group'], ascending = [False, True])
    
    display(df_msa.head(3))
    

In [ ]:
if indicator_name == 'Jobs_2':
    df_jobs2_mpo = df_jobs2_mpo.rename(columns = {'MPO':'Geography'})
    df_jobs2 = pd.concat([df_jobs2_mpo, df_jobs2_nat])
    df_jobs2 = df_jobs2.sort_values(['date_', 'Geography'], ascending = [False, True])

    df_jobs2.columns = [col.lower() for col in df_jobs2.columns]
    df_jobs2.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_jobs2.columns]
    df_jobs2.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_jobs2.columns]
    display(df_jobs2.head(25))


if indicator_name == 'Jobs_3':      
    df_msa  = df_msa.reset_index(drop = True)
    df_msa.columns = [x.lower() for x in df_msa.columns]
    df_msa.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_msa.columns]
    df_msa.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_msa.columns]
    df_msa['date_'] = df_msa['date_'].astype('str')
    display(df_msa.head())


if indicator_name == 'Labor_2':      
    df_msa  = df_msa.reset_index(drop = True)
    df_msa.columns = [x.lower() for x in df_msa.columns]
    df_msa.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_msa.columns]
    df_msa.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_msa.columns]
    display(df_msa.head())





In [ ]:
# Set file path for exporting
path_out_csv  = os.path.join(path_agol, indicator_name)
print('CSV files exported here: ' + path_out_csv)

if indicator_name == 'Jobs_2':
    df_jobs2.to_csv(os.path.join(path_out_csv, 'Jobs_2_MPO_BLS.csv'), index = False)

if indicator_name == 'Labor_2':
    df_msa.to_csv(os.path.join(path_out_csv, 'Labor_2_MSA_BLS_LA.csv'), index = False)